# 1 Calculated

In [1]:
def get_nl_shift(channels, nl_mem, nl_mem_nc1_1, nl_mem_nc1_2, nl_mem_nc2_1, nl_mem_nc2_2, nl_mem_nc3_1, nl_mem_nc3_2, sym_nonlin):
    nl_memory_size = np.zeros(4,dtype=int)
    if sym_nonlin:
        nl_memory_size[0] = nl_mem + 1
    else:
        nl_memory_size[0] = 2 * nl_mem + 1
    nl_memory_size[1] = nl_mem_nc1_1 + nl_mem_nc1_2 + 1
    nl_memory_size[2] = nl_mem_nc2_1 + nl_mem_nc2_2 + 1
    nl_memory_size[3] = nl_mem_nc3_1 + nl_mem_nc3_2 + 1
    
    nl_mem_nc = np.zeros((4,2),dtype=int)
    nl_mem_nc[0,0] = nl_mem
    nl_mem_nc[0,1] = nl_mem
    nl_mem_nc[1,0] = nl_mem_nc1_1
    nl_mem_nc[1,1] = nl_mem_nc1_2
    nl_mem_nc[2,0] = nl_mem_nc2_1
    nl_mem_nc[2,1] = nl_mem_nc2_2
    nl_mem_nc[3,0] = nl_mem_nc3_1
    nl_mem_nc[3,1] = nl_mem_nc3_2

    max_nl_shift = np.max(nl_mem_nc[:channels,:])

    nl_shifts = np.zeros((channels,channels,2),dtype=int)
    for i in range(channels):
        for j in range(channels):
            if i == j:
                nl_shifts[i,i,0] = max_nl_shift - nl_mem
                nl_shifts[i,i,1] = max_nl_shift - nl_mem
            elif i > j:
                nl_shifts[i,j,1] = max_nl_shift - nl_mem_nc[i-j,1]
                nl_shifts[i,j,0] = max_nl_shift - nl_mem_nc[i-j,0]
            else:
                nl_shifts[i,j,1] = max_nl_shift - nl_mem_nc[j-i,0]
                nl_shifts[i,j,0] = max_nl_shift - nl_mem_nc[j-i,1]
                
    return nl_memory_size, nl_shifts, max_nl_shift


def get_steps_shift(num_step, Fn, channel_numbers):
    symbol_shift = (fiberL / num_step * 2 * math.pi * b2 * Fsym * Fn[channel_numbers]).astype(int)    # На сколько символов (сэмплов) сдвигается каждый канал за один шаг
    
    if np.max(symbol_shift) == 0 or not manual_shift:
        step_length = fiberL / num_step
        last_step_length = 0
        FD_step_length = 0
    else:
        if np.any(Fn[channel_numbers] == 0):
            Fn_nonzero = Fn[channel_numbers]
            Fn_nonzero = Fn_nonzero[abs(Fn_nonzero) > 0]
            symbol_shift_distance = np.max(abs(1/(2 * math.pi * b2 * Fsym * Fn_nonzero)))
            ind = np.argmax(abs(1/(2 * math.pi * b2 * Fsym * Fn_nonzero)))                               # Нужно обновить, если индекс правее от удаленного нулевого канала?
            step_length = symbol_shift[ind] * symbol_shift_distance
            symbol_shift = (step_length * 2 * math.pi * b2 * Fsym * Fn[channel_numbers]).astype(int)
            last_symbol_shift = ((fiberL - step_length * num_step) * 2 * math.pi * b2 * Fsym * Fn[channel_numbers]).astype(int) 
            last_step_length = last_symbol_shift[ind] * symbol_shift_distance
            last_symbol_shift = (last_step_length * 2 * math.pi * b2 * Fsym * Fn[channel_numbers]).astype(int)
            FD_step_length = fiberL - step_length * num_step - last_step_length     
        else:
            symbol_shift_distance = np.max(abs(1/(2 * math.pi * b2 * Fsym * Fn[channel_numbers])))       # Максимальное среди всех каналов расстояние, при котором происходит сдвиг на один символ
            ind = np.argmax(abs(1/(2 * math.pi * b2 * Fsym * Fn[channel_numbers])))                      # Номер канала с максимальным расстояние
            step_length = symbol_shift[ind] * symbol_shift_distance                                      # Обновляем длину шага, чтобы за один шаг был сдвиг на целое число символов
            symbol_shift = (step_length * 2 * math.pi * b2 * Fsym * Fn[channel_numbers]).astype(int)     # Для каждого канала обновляем на сколько символов будет сдвиг за один шаг
            last_symbol_shift = ((fiberL - step_length * num_step) * 2 * math.pi * b2 * Fsym * Fn[channel_numbers]).astype(int)   # На сколько символов сдвигается каждый канал за оставшееся расстояние линии связи
            last_step_length = last_symbol_shift[ind] * symbol_shift_distance                            # Определям часть оставшегося расстояния, при котором сдвиг будет на целое число символов
            last_symbol_shift = (last_step_length * 2 * math.pi * b2 * Fsym * Fn[channel_numbers]).astype(int)                    #  Для каждого канала обновляем на сколько символов будет сдвиг за оставшееся расстояние
            FD_step_length = fiberL - step_length * num_step - last_step_length                          # Оставшееся расстояние линии связи со сдвигом менее чем на один символ

    step_shifts = np.zeros((channels,2),dtype=int)
    if last_step_length > 0:
        last_shifts = np.zeros((channels,2),dtype=int)

    for i in range(channels):
        if symbol_shift[i] > 0:
            step_shifts[i,0] = abs(np.min(symbol_shift)) - symbol_shift[i]
            step_shifts[i,1] = np.max(symbol_shift) + symbol_shift[i]
        else:
            step_shifts[i,0] = np.max(symbol_shift) - symbol_shift[i]
            step_shifts[i,1] = abs(np.min(symbol_shift)) + symbol_shift[i]

        if last_step_length > 0:
            if last_symbol_shift[i] > 0:
                last_shifts[i,0] = abs(np.min(last_symbol_shift)) - last_symbol_shift[i]
                last_shifts[i,1] = np.max(last_symbol_shift) + last_symbol_shift[i]
            else:
                last_shifts[i,0] = np.max(last_symbol_shift) - last_symbol_shift[i]
                last_shifts[i,1] = abs(np.min(last_symbol_shift)) + last_symbol_shift[i]

    step_output_shifts = np.zeros((2,),dtype=int)
    full_output_shifts = np.zeros((2,),dtype=int)

    if last_step_length > 0:
        last_output_shifts = np.zeros((2,),dtype=int)


    step_output_shifts[0] = abs(np.min(symbol_shift))
    step_output_shifts[1] = np.max(symbol_shift)  
    if last_step_length > 0:
        last_output_shifts[0] = abs(np.min(last_symbol_shift))
        last_output_shifts[1] = np.max(last_symbol_shift)  

    full_output_shifts[0] = num_step * abs(np.min(symbol_shift))
    full_output_shifts[1] = num_step * np.max(symbol_shift)
    if last_step_length > 0:
        full_output_shifts[0] += abs(np.min(last_symbol_shift))
        full_output_shifts[1] += np.max(last_symbol_shift)

    max_full_shift = int(np.sum(full_output_shifts))
    
    if len(channel_numbers) == 1 and Fn[channel_numbers] == 0:
        return step_length, step_shifts, last_step_length, FD_step_length, step_output_shifts, full_output_shifts, max_full_shift
    else:
        return step_length, step_shifts, last_step_length, last_shifts, last_output_shifts, FD_step_length, step_output_shifts, full_output_shifts, max_full_shift
    

In [2]:
def generate_bits(number_of_bits):
    return np.random.randint(2, size=number_of_bits, dtype=np.uint8)

def bits_to_symbols(bits):
    symbols = np.zeros(int(len(bits)/modulataion_order), dtype=np.complex128)
    for i in range(len(symbols)):
        symbols[i] = gray_symbols_16qam[int(''.join(map(str, tuple(bits[4*i:4*i+4]))), 2)]
    return symbols

def symbols_to_codes(data_complex):
    return np.argmin(abs(data_complex[:, None] - gray_symbols_16qam), axis=1)

# Вычисление коэффициента битовой ошибки BER
def ber_by_codes(tx, rx):
    diff = tx ^ rx
    errors = 0
    for error in diff:
        while error:
            error &= error - 1
            errors +=1
            
    return 0.25 * errors / len(diff)

def ber_by_codes_fast(tx, rx):
    return np.mean(np.unpackbits(np.bitwise_xor(tx, rx).astype(np.uint8)).astype(np.uint8)) * 2

def calculate_ber_from_symbols(symbols_tx, symbols_rx):
    return ber_by_codes(symbols_to_codes(symbols_tx), symbols_to_codes(symbols_rx))

def calculate_ber_from_symbols_fast(symbols_tx, symbols_rx):
    return ber_by_codes_fast(symbols_to_codes(symbols_tx), symbols_to_codes(symbols_rx))

def generate_signal(symbols, rrc, oversampling):
    signal = np.zeros(oversampling * len(symbols), dtype=np.complex128)
    signal[0::oversampling] = symbols
    signal = scipy.signal.fftconvolve(signal, rrc, 'same')

    return signal * np.sqrt(power / np.mean(np.square(np.abs(signal))))

def demodulate_signal_equalizer(signal, symbols_tx, rrc, oversampling, shift=1):
    s_eq = np.zeros((int(len(signal)/oversampling), 2), dtype=np.float64)
    
    symbols = scipy.signal.fftconvolve(signal, rrc, 'same')[shift::oversampling]
    s_eq[:,0] = np.real(symbols)
    s_eq[:,1] = np.imag(symbols)
    coeff = np.matmul(np.linalg.pinv(np.matmul(s_eq.conj().T, s_eq)),s_eq.conj().T).dot(symbols_tx)
    symbols_eq = s_eq.dot(coeff)
    symbols_eq /= np.sqrt(np.mean(np.square(np.abs(symbols_eq))))
    return symbols_eq 


def demodulate_signal_phase(signal, symbols_tx, rrc, oversampling, shift=1):
    if oversampling == 1:
        shift = 0
    
    s_eq = np.zeros((int(len(signal)/oversampling), 2), dtype=np.float64)
    
    symbols = scipy.signal.fftconvolve(signal, rrc, 'same')[shift::oversampling]

    ber_min = 1
    phase, phase_cur, phase_step = 0, 0, np.pi/4
    for k in range(5):
        for l in range(8):
            symbols_rotated = symbols * np.exp(-1j * (phase_cur + l * phase_step))
            symbols_rotated /= np.sqrt(np.mean(np.square(np.abs(symbols_rotated))))
            ber = calculate_ber_from_symbols_fast(symbols_tx, symbols_rotated)
            if ber < ber_min:
                ber_min = ber
                phase = phase_cur + l * phase_step
    
        phase_cur = phase - phase_step
        phase_step /= 4

    symbols_rotated = symbols * np.exp(-1j * phase)
    symbols_rotated /= np.sqrt(np.mean(np.square(np.abs(symbols_rotated))))
    
    return symbols_rotated, ber_min


def demodulate_signal_phase_1sps(symbols, symbols_tx):
    
    s_eq = np.zeros((len(symbols), 2), dtype=np.float64)
    
    ber_min = 1
    phase, phase_cur, phase_step = 0, 0, np.pi/4
    for k in range(5):
        for l in range(8):
            symbols_rotated = symbols * np.exp(-1j * (phase_cur + l * phase_step))
            symbols_rotated /= np.sqrt(np.mean(np.square(np.abs(symbols_rotated))))
            ber = calculate_ber_from_symbols_fast(symbols_tx, symbols_rotated)
            if ber < ber_min:
                ber_min = ber
                phase = phase_cur + l * phase_step
    
        phase_cur = phase - phase_step
        phase_step /= 4

    symbols_rotated = symbols * np.exp(-1j * phase)
    symbols_rotated /= np.sqrt(np.mean(np.square(np.abs(symbols_rotated))))
    
    return symbols_rotated, ber_min

def get_rrc_filter(oversampling, rrc_width, roll_off):
    coefficients = np.zeros(rrc_width * oversampling)
    mid = int(rrc_width * oversampling / 2)

    for i in range(rrc_width * oversampling):
        if i == mid:
            coefficients[i] = (1 - roll_off + 4. * roll_off / math.pi)
        elif abs(1.0 - 16. * roll_off * roll_off * (i - mid) * (i - mid) / (oversampling * oversampling)) < 1e-10:
            coefficients[i] = roll_off * ((1. + 2. / math.pi) * np.sin(math.pi / (4. * roll_off)) + (1. - 2. / math.pi)
                                          * np.cos(math.pi / (4. * roll_off))) / np.sqrt(2)
        else:
            coefficients[i] = (np.sin(math.pi * (i - mid) / oversampling * (1 - roll_off)) + 4 * roll_off * (i - mid) / oversampling 
                               * np.cos(math.pi * (i - mid) / oversampling * (1 + roll_off))) \
                              / (math.pi * (i - mid) / oversampling 
                                 * (1. - 16. * roll_off * roll_off * (i - mid) * (i - mid) / oversampling / oversampling))
    
    return 2 * coefficients / np.sum((abs(coefficients)))


def cd_operator(data, length, oversampling, chFreq, direction):
    size = len(data)
    dw = 2 * math.pi * symbol_rate * oversampling / size
    w = np.arange(-size/2, size/2,1) * dw
    w = np.fft.fftshift(w)
    
    if direction == 'f':
        s = 1
    if direction == 'b':
        s = -1
    
    fft_data = np.fft.fft(data)
    fft_data = fft_data * np.exp(s * 1j * b_2 / 2 * (w + 2 * math.pi * chFreq) ** 2 * length)
    dataCD = np.fft.ifft(fft_data)

    return dataCD

def ber_to_qfactor(ber):
    return 20 * np.log10(np.sqrt(10) * scipy.special.erfcinv(8 / 3 * ber));

# 2 BER and CD

In [3]:
# Классификация отсчётов комплексной амплитуды
def symbols_to_codes_v2(data_complex):
    codes = np.zeros(len(data_complex), dtype=np.int32)
    for i in range(len(data_complex)):
        codes[i] = np.argmin(abs(gray_symbols_16qam - data_complex[i]))
    
    return codes

# # Вычисление коэффициента битовой ошибки BER
# def ber_by_codes(tx, rx):
#     diff = tx ^ rx
#     errors = 0
#     for error in diff:
#         while error:
#             error &= error - 1
#             errors +=1
            
#     return 0.25 * errors / len(diff)

def calculate_ber(tx, rx):
    return ber_by_codes(symbols_to_codes_v2(tx[0,0,:] + 1j * tx[0,1,:]), symbols_to_codes_v2(rx[0,0,:] + 1j * rx[0,1,:]))

def demapper(tx, rx):
    phi = (1+np.sqrt(5))/2
    xl = 0.9
    xr = 1.1
    for j in range(10):
        x1 = xr - (xr-xl)/phi
        x2 = xl + (xr-xl)/phi
        y1 = calculate_ber(tx, rx * x1)
        y2 = calculate_ber(tx, rx * x2)
        if y1 >= y2:
            xl = x1
        else:
            xr = x2
        x = (x1+x2)/2
    
    return x

# Восстановление или компенсация дисперсионного уширения сигнала
# def cd_operator(data, size, length, chFreq, direction):
#     dw = 2*math.pi*Fsym/size
#     w = np.arange(-size/2, size/2,1) * dw
#     w = np.fft.fftshift(w)
    
#     if direction == 'f':
#         s = 1
#     if direction == 'b':
#         s = -1
    
#     fft_data = np.fft.fft(data)
#     fft_data = fft_data * np.exp(s * 1j * b2 / 2 * (w + 2 * math.pi * chFreq) ** 2 * length)
#     dataCD = np.fft.ifft(fft_data)

#     return dataCD

def create_metadata(channels, channel_numbers, train_portion, train_data_size, CDC_data_size, num_step, filt_CDC_width, filt_CDC_last_width, filt_FD_width, nl_mem, nl_mem_nc1_1, nl_mem_nc1_2, nl_mem_nc2_1, nl_mem_nc2_2, nl_mem_nc3_1, nl_mem_nc3_2,
                   epochs, batch_size, lrate, lrate_CDC, manual_shift, sym_filt, sym_nonlin, save_model_best):
    metadata1 = ['channels', 'channel_numbers', 'train_portion', 'train_data_size', 'CDC_data_size', 'num_step', 'filt_CDC_width', 'nl_mem', 'nl_mem_nc1_1', 'nl_mem_nc1_2', 'nl_mem_nc2_1', 'nl_mem_nc2_2', 
                 'nl_mem_nc3_1', 'nl_mem_nc3_2']
    metadata2 = ['epochs', 'batch_size','lrate', 'lrate_CDC', 'manual_shift', 'sym_filt', 'sym_nonlin', 'save_model_best']
    #dict( (name, eval(name)) for name in ['channels', 'channel_numbers', 'train_portion', 'train_data_size', 
    #                                                  'CDC_data_size', 'num_step', 'filt_CDC_width', 'nl_mem', 'nl_mem_nc1_1', 
    #                                                  'nl_mem_nc1_2', 'nl_mem_nc2_1', 'nl_mem_nc2_2', 'nl_mem_nc3_1', 'nl_mem_nc3_2'])
    #metadata2 = dict( (name, eval(name)) for name in ['epochs', 'batch_size','lrate', 'lrate_CDC', 'manual_shift', 'sym_filt', 
    #                                                  'sym_nonlin', 'save_model_best'])

    now = datetime.datetime.now()
    date_and_time = now.strftime('%Y%m%d_%H%M')
    dir_path = results_path + date_and_time + '/'
    makedirs(dir_path)
    
    f = open(dir_path + 'metadata.md', 'w')
    f.write(now.strftime('%Y-%m-%d %H:%M') + '\n\n')
    
    for l in metadata1:
        f.write('{}: {}\n'.format(l, eval(l)))
        #f.write('{}: {}\n'.format(x, y))
    if manual_shift:
        f.write('filt_CDC_last_width: {}\n'.format(filt_CDC_last_width))
        f.write('filt_FD_width: {}\n'.format(filt_FD_width))
    for l in metadata2:
        f.write('{}: {}\n'.format(l, eval(l)))
        #f.write('{}: {}\n'.format(x, y))
    f.close()
    
    return dir_path

# 3 Initialization and Dataloader

In [4]:
# Определение класса для инициализации весов сверточных слоев заданным массивом
class CustomInit(mx.init.Initializer):
    def __init__(self, filt):
        super(CustomInit, self).__init__()
        self.filt = filt
        
    def _init_weight(self, _, arr):
        arr[:] = self.filt

# Разделение данных на батчи
def dataloader(X, y, batch, channels, data_dimension, full_delay, full_output_shifts, max_full_shift, enableShuffle, pol=2):
    size = X.shape[1]
    batch = min(batch, size)
    d = size / batch
    if d%1 > 0.2:
        batch = np.floor(size / np.ceil(d)).astype(int)
    batch = batch - batch%2
    number_of_bathes = np.floor(size / batch).astype(int)
    
    data = nd.empty((number_of_bathes, data_dimension, batch))
    if manual_shift:
        label = nd.empty((number_of_bathes, data_dimension, batch-2*full_delay-max_full_shift))
    else:
        label = nd.empty((number_of_bathes, data_dimension, batch-2*full_delay))
    for i in range(number_of_bathes):
        data[i,:,:] = nd.array(X[:,i*batch:(i+1)*batch])
        for j in range(channels):
            if manual_shift:
                label[i,j*2*pol:(j+1)*2*pol,:] = nd.array(y[j*2*pol:(j+1)*2*pol,i*batch+full_delay+full_output_shifts[0]:(i+1)*batch-full_delay-full_output_shifts[1]])
            else:
                label[i,j*2*pol:(j+1)*2*pol,:] = nd.array(y[j*2*pol:(j+1)*2*pol,i*batch+full_delay:(i+1)*batch-full_delay])

    dataset = gluon.data.dataset.ArrayDataset(data, label)
    return gluon.data.DataLoader(dataset, batch_size=1, shuffle=enableShuffle), batch, number_of_bathes

NameError: name 'mx' is not defined

# 4 Linear and Nonlinear layers

In [ ]:
# Комплексный свёрточный слой со случайной инициализацией весов
class ComplexConvRandom(gluon.Block):
    def __init__(self, kernel_size, dimension, shift=np.array([[0,0]]), pol=2, enable_shift=True, sym=False, channels=1, strides=1, **kwargs):
        super(ComplexConvRandom, self).__init__(**kwargs)
        with self.name_scope():
            if len(shift) == 1 and np.sum(shift[0,:]) == 0:
                    shift = np.zeros((int(dimension/(2*pol)), 2),dtype=int)
            self.dimension = dimension
            self.shift = shift
            self.pol = pol
            self.kernel_size = kernel_size
            self.enable_shift = enable_shift
            self.sym = sym
            self.weight_re = self.params.get('weight_re', allow_deferred_init=True, shape=(int(dimension/(2*pol)),1,kernel_size))
            self.weight_im = self.params.get('weight_im', allow_deferred_init=True, shape=(int(dimension/(2*pol)),1,kernel_size))

    def forward(self, z):
        with z.context:
            chan = int(self.dimension/(2*self.pol))
            
            if self.sym:
                filt_re = nd.concat(self.weight_re.data(), nd.flip(self.weight_re.data(), axis=2)[:,:,1:], dim=2)
                filt_im = nd.concat(self.weight_im.data(), nd.flip(self.weight_im.data(), axis=2)[:,:,1:], dim=2)
                kernel = 2 * self.kernel_size - 1
            else:
                filt_re = self.weight_re.data()
                filt_im = self.weight_im.data()
                kernel = self.kernel_size
            
            y = nd.Convolution(data = z[:,0::2*self.pol,:], weight = filt_re, no_bias=True, num_filter=chan, kernel=kernel, num_group=chan) -\
            nd.Convolution(data = z[:,1::2*self.pol,:], weight = filt_im, no_bias=True, num_filter=chan, kernel=kernel, num_group=chan)
           
            y = nd.concat(y, nd.Convolution(data = z[:,0::2*self.pol,:], weight = filt_im, no_bias=True, num_filter=chan, kernel=kernel, num_group=chan) + 
                          nd.Convolution(data = z[:,1::2*self.pol,:], weight = filt_re, no_bias=True, num_filter=chan, kernel=kernel, num_group=chan), dim=1)
            
            for i in range(2, 2*self.pol, 2):
                y = nd.concat(y, nd.Convolution(data = z[:,i::2*self.pol,:], weight = filt_re, no_bias=True, num_filter=chan, kernel=kernel, num_group=chan) - 
                          nd.Convolution(data = z[:,i+1::2*self.pol,:], weight = filt_im, no_bias=True, num_filter=chan, kernel=kernel, num_group=chan), dim=1)
                y = nd.concat(y, nd.Convolution(data = z[:,i::2*self.pol,:], weight = filt_im, no_bias=True, num_filter=chan, kernel=kernel, num_group=chan) + 
                          nd.Convolution(data = z[:,i+1::2*self.pol,:], weight = filt_re, no_bias=True, num_filter=chan, kernel=kernel, num_group=chan), dim=1)

            v = nd.slice(y, begin=(None, 0, None), end=(None, self.dimension, None), step=(1, chan, 1))
            if self.enable_shift:
                v = v[:,:,self.shift[0,0]:v.shape[2]-self.shift[0,1]]
            for i in range(1, chan):
                u = nd.slice(y, begin=(None, i, None), end=(None, self.dimension, None), step=(1, chan, 1))
                if self.enable_shift:
                    u = u[:,:,self.shift[i,0]:u.shape[2]-self.shift[i,1]]
                v = nd.concat(v, u, dim=1)
            return v
        
# Комплексный свёрточный слой с предопределёнными весами
class ComplexConvPredefined(gluon.Block):
    def __init__(self, weightsRe, weightsIm, kernel_size, dimension, shift=np.array([[0,0]]), pol=2, enable_shift=True, sym=False, channels=1, strides=1, **kwargs):
        super(ComplexConvPredefined, self).__init__(**kwargs)
        with self.name_scope():
            if len(shift) == 1 and np.sum(shift[0,:]) == 0:
                    shift = np.zeros((int(dimension/(2*pol)), 2),dtype=int)
            self.dimension = dimension
            self.shift = shift
            self.pol = pol
            self.kernel_size = kernel_size
            self.enable_shift = enable_shift
            self.sym = sym
            self.weight_re = self.params.get('weight_re', allow_deferred_init=True, shape=(int(dimension/(2*pol)),1,kernel_size), init=CustomInit(weightsRe))
            self.weight_im = self.params.get('weight_im', allow_deferred_init=True, shape=(int(dimension/(2*pol)),1,kernel_size), init=CustomInit(weightsIm))

    def forward(self, z):
        with z.context:
            chan = int(self.dimension/(2*self.pol))
            
            if self.sym:
                filt_re = nd.concat(self.weight_re.data(), nd.flip(self.weight_re.data(), axis=2)[:,:,1:], dim=2)
                filt_im = nd.concat(self.weight_im.data(), nd.flip(self.weight_im.data(), axis=2)[:,:,1:], dim=2)
                kernel = 2 * self.kernel_size - 1
            else:
                filt_re = self.weight_re.data()
                filt_im = self.weight_im.data()
                kernel = self.kernel_size
            
            y = nd.Convolution(data = z[:,0::2*self.pol,:], weight = filt_re, no_bias=True, num_filter=chan, kernel=kernel, num_group=chan) -\
            nd.Convolution(data = z[:,1::2*self.pol,:], weight = filt_im, no_bias=True, num_filter=chan, kernel=kernel, num_group=chan)
           
            y = nd.concat(y, nd.Convolution(data = z[:,0::2*self.pol,:], weight = filt_im, no_bias=True, num_filter=chan, kernel=kernel, num_group=chan) + 
                          nd.Convolution(data = z[:,1::2*self.pol,:], weight = filt_re, no_bias=True, num_filter=chan, kernel=kernel, num_group=chan), dim=1)
            
            for i in range(2, 2*self.pol, 2):
                y = nd.concat(y, nd.Convolution(data = z[:,i::2*self.pol,:], weight = filt_re, no_bias=True, num_filter=chan, kernel=kernel, num_group=chan) - 
                          nd.Convolution(data = z[:,i+1::2*self.pol,:], weight = filt_im, no_bias=True, num_filter=chan, kernel=kernel, num_group=chan), dim=1)
                y = nd.concat(y, nd.Convolution(data = z[:,i::2*self.pol,:], weight = filt_im, no_bias=True, num_filter=chan, kernel=kernel, num_group=chan) + 
                          nd.Convolution(data = z[:,i+1::2*self.pol,:], weight = filt_re, no_bias=True, num_filter=chan, kernel=kernel, num_group=chan), dim=1)
            
            v = nd.slice(y, begin=(None, 0, None), end=(None, self.dimension, None), step=(1, chan, 1))
            if self.enable_shift:
                v = v[:,:,self.shift[0,0]:v.shape[2]-self.shift[0,1]]
            for i in range(1, chan):
                u = nd.slice(y, begin=(None, i, None), end=(None, self.dimension, None), step=(1, chan, 1))
                if self.enable_shift:
                    u = u[:,:,self.shift[i,0]:u.shape[2]-self.shift[i,1]]
                v = nd.concat(v, u, dim=1)
            
            return v
        
class KerrActivationEnhanced_v2_1ch(gluon.Block): # Добавить сдвиги для случая разных коэффициентов для поляризаций и разная ширина на разных каналов
    def __init__(self, dimension, tensor_intra, nl_shifts, nl_memory_size, max_nl_shift, nl_coef=0.1, pol=2, strides=1, **kwargs):
        super(KerrActivationEnhanced_v2_1ch, self).__init__()
        chan = 1
        self.chan = chan
        self.nl_shifts = nl_shifts
        self.pol = pol
        self.nl_memory_size = nl_memory_size
        self.max_nl_shift = max_nl_shift
        with self.name_scope():
            self.conv_intra = self.params.get('conv_intra', allow_deferred_init=True, shape=(chan,1,nl_memory_size[0]), init=CustomInit(tensor_intra))
            self.gamma = self.params.get('gamma', shape=(1,), allow_deferred_init=True, init=mx.init.Constant(nl_coef))

    def forward(self, z):
        if self.pol == 1:
            power = nd.square(z[:,0:1,:]) +  nd.square(z[:,1:2,:])
        if self.pol == 2:
            power = nd.square(z[:,0:1,:]) +  nd.square(z[:,1:2,:]) + nd.square(z[:,2:3,:]) +  nd.square(z[:,3:4,:])
        power_intra = power[:,:,self.nl_shifts[0,0,0]:power.shape[2]-self.nl_shifts[0,0,1]]
                
        if sym_nonlin:
            tens_intra = nd.concat(self.conv_intra.data(), nd.flip(self.conv_intra.data(), axis=2)[:,:,1:], dim=2)
            power = nd.Convolution(data = power_intra, weight = tens_intra, no_bias=True, num_filter=self.chan, kernel=2*self.nl_memory_size[0]-1, num_group=self.chan)
        else:
            power = nd.Convolution(data = power_intra, weight = self.conv_intra.data(), no_bias=True, num_filter=self.chan, kernel=self.nl_memory_size[0], num_group=self.chan)
        # power = self.gamma.data() * power          
        u = nd.cos(self.gamma.data() * power) * z[:,0,self.max_nl_shift:z.shape[2]-self.max_nl_shift] + nd.sin(self.gamma.data() * power) * z[:,1,self.max_nl_shift:z.shape[2]-self.max_nl_shift]
        u = nd.concat(u, nd.cos(self.gamma.data() * power) * z[:,1,self.max_nl_shift:z.shape[2]-self.max_nl_shift] - nd.sin(self.gamma.data() * power) * z[:,0,self.max_nl_shift:z.shape[2]-self.max_nl_shift], dim=1)
        if self.pol == 2:
            u = nd.concat(u, nd.cos(self.gamma.data() * power) * z[:,2,self.max_nl_shift:z.shape[2]-self.max_nl_shift] + nd.sin(self.gamma.data() * power) * z[:,3,self.max_nl_shift:z.shape[2]-self.max_nl_shift], dim=1)
            u = nd.concat(u, nd.cos(self.gamma.data() * power) * z[:,3,self.max_nl_shift:z.shape[2]-self.max_nl_shift] - nd.sin(self.gamma.data() * power) * z[:,2,self.max_nl_shift:z.shape[2]-self.max_nl_shift], dim=1)

        return u
    
class KerrActivationEnhanced_v2_2ch(gluon.Block): # Добавить сдвиги для случая разных коэффициентов для поляризаций и разная ширина на разных каналов
    def __init__(self, dimension, tensor_intra, tensor_inter, nl_shifts, nl_coef=0.1, pol=2, strides=1, **kwargs):
        super(KerrActivationEnhanced_v2_2ch, self).__init__()
        chan = int(dimension/4)
        self.chan = chan
        self.nl_shifts = nl_shifts
        with self.name_scope():
            self.conv_intra = self.params.get('conv_intra', allow_deferred_init=True, shape=(chan,1,nl_memory_size[0]), init=CustomInit(tensor_intra))
            self.conv_inter = gluon.nn.Conv1D(channels=2, in_channels=2, groups=2, kernel_size=nl_memory_size[1], strides=strides, activation=None, use_bias=False, weight_initializer=CustomInit(tensor_inter))
            self.gamma = self.params.get('gamma', shape=(1,), allow_deferred_init=True, init=mx.init.Constant(nl_coef))

    def forward(self, z):
        power = nd.square(z[:,0:1,:]) +  nd.square(z[:,1:2,:]) + nd.square(z[:,2:3,:]) +  nd.square(z[:,3:4,:])
        power_intra = power[:,:,self.nl_shifts[0,0,0]:power.shape[2]-self.nl_shifts[0,0,1]]
        power_inter = power[:,:,self.nl_shifts[0,1,0]:power.shape[2]-self.nl_shifts[0,1,1]]
                
        power = nd.square(z[:,4:5,:]) +  nd.square(z[:,5:6,:]) + nd.square(z[:,6:7,:]) +  nd.square(z[:,7:8,:])
        power_intra = nd.concat(power_intra, power[:,:,self.nl_shifts[1,1,0]:power.shape[2]-self.nl_shifts[1,1,1]], dim=1)
        power_inter = nd.concat(power_inter, power[:,:,self.nl_shifts[1,0,0]:power.shape[2]-self.nl_shifts[1,0,1]], dim=1)

        if sym_nonlin:
            tens_intra = nd.concat(self.conv_intra.data(), nd.flip(self.conv_intra.data(), axis=2)[:,:,1:], dim=2)
            power_intra = nd.Convolution(data = power_intra, weight = tens_intra, no_bias=True, num_filter=self.chan, kernel=2*nl_memory_size[0]-1, num_group=self.chan)
        else:
            power_intra = nd.Convolution(data = power_intra, weight = self.conv_intra.data(), no_bias=True, num_filter=self.chan, kernel=nl_memory_size[0], num_group=self.chan)
        power_inter = self.conv_inter(power_inter)
        power = power_intra + nd.concat(power_inter[:,1:2,:], power_inter[:,0:1,:])
                       
        u = nd.cos(self.gamma.data() * power[:,0:1,:]) * z[:,0,max_nl_shift:z.shape[2]-max_nl_shift] + nd.sin(self.gamma.data() * power[:,0:1,:]) * z[:,1,max_nl_shift:z.shape[2]-max_nl_shift]
        u = nd.concat(u, nd.cos(self.gamma.data() * power[:,0:1,:]) * z[:,1,max_nl_shift:z.shape[2]-max_nl_shift] - nd.sin(self.gamma.data() * power[:,0:1,:]) * z[:,0,max_nl_shift:z.shape[2]-max_nl_shift], dim=1)
        for i in range(1, self.chan * 2):
            ind = int(i / 2)
            u = nd.concat(u, nd.cos(self.gamma.data() * power[:,ind:ind+1,:]) * z[:,2*i,max_nl_shift:z.shape[2]-max_nl_shift] + nd.sin(self.gamma.data() * power[:,ind:ind+1,:]) * z[:,2*i+1,max_nl_shift:z.shape[2]-max_nl_shift], dim=1)
            u = nd.concat(u, nd.cos(self.gamma.data() * power[:,ind:ind+1,:]) * z[:,2*i+1,max_nl_shift:z.shape[2]-max_nl_shift] - nd.sin(self.gamma.data() * power[:,ind:ind+1,:]) * z[:,2*i,max_nl_shift:z.shape[2]-max_nl_shift], dim=1)

        return u
    
    
class KerrActivationEnhanced_v2_2ch_LS_0(gluon.Block): # Добавить сдвиги для случая разных коэффициентов для поляризаций и разная ширина на разных каналов
    def __init__(self, dimension, tensor_intra, tensor_inter, nl_shifts, nl_coef=0.1, pol=2, strides=1, **kwargs):
        super(KerrActivationEnhanced_v2_2ch_LS_0, self).__init__()
        chan = int(dimension/4)
        self.chan = chan
        self.nl_shifts = nl_shifts
        with self.name_scope():
            self.conv_intra = self.params.get('conv_intra', allow_deferred_init=True, shape=(chan,1,nl_memory_size[0]), init=CustomInit(tensor_intra))
            self.conv_inter = gluon.nn.Conv1D(channels=2, in_channels=2, groups=2, kernel_size=nl_memory_size[1], strides=1, activation=None, use_bias=False, weight_initializer=CustomInit(tensor_inter))
            # добавить выше padding, dilation или strides, чтобы не учитывать одни и те же символы
            self.conv_inter_ds = gluon.nn.Conv1D(channels=1, kernel_size=2, strides=2, activation=None, use_bias=False, weight_initializer=CustomInit(nd.array([0.5, 0.5])))
            self.conv_inter_us = gluon.nn.Conv1DTranspose(channels=1, kernel_size=2, strides=2, activation=None, use_bias=False, weight_initializer=CustomInit(nd.array([1, 1])))
            # self.conv_inter_ds = gluon.nn.Conv1D(channels=1, kernel_size=10, strides=10, activation=None, use_bias=False, weight_initializer=CustomInit(nd.array([0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1])))
            # self.conv_inter_us = gluon.nn.Conv1DTranspose(channels=1, kernel_size=10, strides=10, activation=None, use_bias=False, weight_initializer=CustomInit(nd.array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1])))
            self.gamma = self.params.get('gamma', shape=(1,), allow_deferred_init=True, init=mx.init.Constant(nl_coef))

    def forward(self, z):
        power = nd.square(z[:,0:1,:]) +  nd.square(z[:,1:2,:]) + nd.square(z[:,2:3,:]) +  nd.square(z[:,3:4,:])
        power_intra = power[:,:,self.nl_shifts[0,0,0]:power.shape[2]-self.nl_shifts[0,0,1]]
        
        power_inter = self.conv_inter_ds(power[:,:,self.nl_shifts[0,1,0]:power.shape[2]-self.nl_shifts[0,1,1]])
        
        power_inter = self.conv_inter_us(power_inter)
        
        
        power = nd.square(z[:,4:5,:]) +  nd.square(z[:,5:6,:]) + nd.square(z[:,6:7,:]) +  nd.square(z[:,7:8,:])
        power_intra = nd.concat(power_intra, power[:,:,self.nl_shifts[1,1,0]:power.shape[2]-self.nl_shifts[1,1,1]], dim=1)
        power_inter_0 = self.conv_inter_ds(power[:,:,self.nl_shifts[1,0,0]:power.shape[2]-self.nl_shifts[1,0,1]])
        power_inter_0 = self.conv_inter_us(power_inter_0)
        power_inter = nd.concat(power_inter, power_inter_0, dim=1)

        if sym_nonlin:
            tens_intra = nd.concat(self.conv_intra.data(), nd.flip(self.conv_intra.data(), axis=2)[:,:,1:], dim=2)
            power_intra = nd.Convolution(data = power_intra, weight = tens_intra, no_bias=True, num_filter=self.chan, kernel=2*nl_memory_size[0]-1, num_group=self.chan)
        else:
            power_intra = nd.Convolution(data = power_intra, weight = self.conv_intra.data(), no_bias=True, num_filter=self.chan, kernel=nl_memory_size[0], num_group=self.chan)
        
        
        power_inter = self.conv_inter(power_inter)
        
        power = power_intra + nd.concat(power_inter[:,1:2,:], power_inter[:,0:1,:])
                       
        u = nd.cos(self.gamma.data() * power[:,0:1,:]) * z[:,0,max_nl_shift:z.shape[2]-max_nl_shift] + nd.sin(self.gamma.data() * power[:,0:1,:]) * z[:,1,max_nl_shift:z.shape[2]-max_nl_shift]
        u = nd.concat(u, nd.cos(self.gamma.data() * power[:,0:1,:]) * z[:,1,max_nl_shift:z.shape[2]-max_nl_shift] - nd.sin(self.gamma.data() * power[:,0:1,:]) * z[:,0,max_nl_shift:z.shape[2]-max_nl_shift], dim=1)
        for i in range(1, self.chan * 2):
            ind = int(i / 2)
            u = nd.concat(u, nd.cos(self.gamma.data() * power[:,ind:ind+1,:]) * z[:,2*i,max_nl_shift:z.shape[2]-max_nl_shift] + nd.sin(self.gamma.data() * power[:,ind:ind+1,:]) * z[:,2*i+1,max_nl_shift:z.shape[2]-max_nl_shift], dim=1)
            u = nd.concat(u, nd.cos(self.gamma.data() * power[:,ind:ind+1,:]) * z[:,2*i+1,max_nl_shift:z.shape[2]-max_nl_shift] - nd.sin(self.gamma.data() * power[:,ind:ind+1,:]) * z[:,2*i,max_nl_shift:z.shape[2]-max_nl_shift], dim=1)

        return u
    
    
class KerrActivationEnhanced_v2_2ch_SPM(gluon.Block): # Добавить сдвиги для случая разных коэффициентов для поляризаций и разная ширина на разных каналов
    def __init__(self, dimension, tensor_intra, nl_shifts, nl_coef=0.1, pol=2, strides=1, **kwargs):
        super(KerrActivationEnhanced_v2_2ch_SPM, self).__init__()
        chan = int(dimension/4)
        self.chan = chan
        self.nl_shifts = nl_shifts
        with self.name_scope():
            self.conv_intra = self.params.get('conv_intra', allow_deferred_init=True, shape=(chan,1,nl_memory_size[0]), init=CustomInit(tensor_intra))
            self.gamma = self.params.get('gamma', shape=(1,), allow_deferred_init=True, init=mx.init.Constant(nl_coef))

    def forward(self, z):
        power = nd.square(z[:,0:1,:]) +  nd.square(z[:,1:2,:]) + nd.square(z[:,2:3,:]) +  nd.square(z[:,3:4,:])
        power_intra = power[:,:,self.nl_shifts[0,0,0]:power.shape[2]-self.nl_shifts[0,0,1]]
                
        power = nd.square(z[:,4:5,:]) +  nd.square(z[:,5:6,:]) + nd.square(z[:,6:7,:]) +  nd.square(z[:,7:8,:])
        power_intra = nd.concat(power_intra, power[:,:,self.nl_shifts[1,1,0]:power.shape[2]-self.nl_shifts[1,1,1]], dim=1)

        if sym_nonlin:
            tens_intra = nd.concat(self.conv_intra.data(), nd.flip(self.conv_intra.data(), axis=2)[:,:,1:], dim=2)
            power_intra = nd.Convolution(data = power_intra, weight = tens_intra, no_bias=True, num_filter=self.chan, kernel=2*nl_memory_size[0]-1, num_group=self.chan)
        else:
            power_intra = nd.Convolution(data = power_intra, weight = self.conv_intra.data(), no_bias=True, num_filter=self.chan, kernel=nl_memory_size[0], num_group=self.chan)

        power = power_intra
                       
        u = nd.cos(self.gamma.data() * power[:,0:1,:]) * z[:,0,max_nl_shift:z.shape[2]-max_nl_shift] + nd.sin(self.gamma.data() * power[:,0:1,:]) * z[:,1,max_nl_shift:z.shape[2]-max_nl_shift]
        u = nd.concat(u, nd.cos(self.gamma.data() * power[:,0:1,:]) * z[:,1,max_nl_shift:z.shape[2]-max_nl_shift] - nd.sin(self.gamma.data() * power[:,0:1,:]) * z[:,0,max_nl_shift:z.shape[2]-max_nl_shift], dim=1)
        for i in range(1, self.chan * 2):
            ind = int(i / 2)
            u = nd.concat(u, nd.cos(self.gamma.data() * power[:,ind:ind+1,:]) * z[:,2*i,max_nl_shift:z.shape[2]-max_nl_shift] + nd.sin(self.gamma.data() * power[:,ind:ind+1,:]) * z[:,2*i+1,max_nl_shift:z.shape[2]-max_nl_shift], dim=1)
            u = nd.concat(u, nd.cos(self.gamma.data() * power[:,ind:ind+1,:]) * z[:,2*i+1,max_nl_shift:z.shape[2]-max_nl_shift] - nd.sin(self.gamma.data() * power[:,ind:ind+1,:]) * z[:,2*i,max_nl_shift:z.shape[2]-max_nl_shift], dim=1)

        return u

    
class KerrActivationEnhanced_v2_3ch(gluon.Block): # Добавить сдвиги для случая разных коэффициентов для поляризаций и разная ширина на разных каналов
    def __init__(self, dimension, tensor_intra, tensor_inter1, tensor_inter2, nl_shifts, nl_coef=0.1, pol=2, strides=1, **kwargs):
        super(KerrActivationEnhanced_v2_3ch, self).__init__()
        chan = int(dimension/4)
        self.chan = chan
        self.nl_shifts = nl_shifts
        with self.name_scope():
            self.conv_intra = self.params.get('conv_intra', allow_deferred_init=True, shape=(chan,1,nl_memory_size[0]), init=CustomInit(tensor_intra))
            self.conv_inter1 = gluon.nn.Conv1D(channels=4, in_channels=4, groups=4, kernel_size=nl_memory_size[1], strides=strides, activation=None, use_bias=False, weight_initializer=CustomInit(tensor_inter1))
            self.conv_inter2 = gluon.nn.Conv1D(channels=2, in_channels=2, groups=2, kernel_size=nl_memory_size[2], strides=strides, activation=None, use_bias=False, weight_initializer=CustomInit(tensor_inter2))
            self.gamma = self.params.get('gamma', shape=(1,), allow_deferred_init=True, init=mx.init.Constant(nl_coef))

    def forward(self, z):
        power = nd.square(z[:,0:1,:]) +  nd.square(z[:,1:2,:]) + nd.square(z[:,2:3,:]) +  nd.square(z[:,3:4,:])
        power_intra = power[:,:,self.nl_shifts[0,0,0]:power.shape[2]-self.nl_shifts[0,0,1]]
        power_inter1 = power[:,:,self.nl_shifts[0,1,0]:power.shape[2]-self.nl_shifts[0,1,1]]
        power_inter2 = power[:,:,self.nl_shifts[0,2,0]:power.shape[2]-self.nl_shifts[0,2,1]]
                
        for i in range(1, self.chan):
            power = nd.square(z[:,4*i:4*i+1,:]) +  nd.square(z[:,4*i+1:4*i+2,:]) + nd.square(z[:,4*i+2:4*i+3,:]) +  nd.square(z[:,4*i+3:4*i+4,:])
            for j in range(self.chan):
                if i == j:
                    power_intra = nd.concat(power_intra, power[:,:,self.nl_shifts[i,i,0]:power.shape[2]-self.nl_shifts[i,i,1]], dim=1)
                elif abs(i-j) == 1:
                    power_inter1 = nd.concat(power_inter1, power[:,:,self.nl_shifts[i,j,0]:power.shape[2]-self.nl_shifts[i,j,1]], dim=1)
                elif abs(i-j) == 2:
                    power_inter2 = nd.concat(power_inter2, power[:,:,self.nl_shifts[i,j,0]:power.shape[2]-self.nl_shifts[i,j,1]], dim=1)

        if sym_nonlin:
            tens_intra = nd.concat(self.conv_intra.data(), nd.flip(self.conv_intra.data(), axis=2)[:,:,1:], dim=2)
            power_intra = nd.Convolution(data = power_intra, weight = tens_intra, no_bias=True, num_filter=self.chan, kernel=2*nl_memory_size[0]-1, num_group=self.chan)
        else:
            power_intra = nd.Convolution(data = power_intra, weight = self.conv_intra.data(), no_bias=True, num_filter=self.chan, kernel=nl_memory_size[0], num_group=self.chan)
        power_inter1 = self.conv_inter1(power_inter1)
        power_inter2 = self.conv_inter2(power_inter2)
        power_inter = [power_inter1, power_inter2]
        
        power = power_intra
        power = power + nd.concat(power_inter1[:,1:2,:], power_inter1[:,0:1,:], power_inter2[:,0:1,:])
        power = power + nd.concat(power_inter2[:,1:2,:], power_inter1[:,2:3,:], power_inter1[:,1:2,:])
                               
        u = nd.cos(self.gamma.data() * power[:,0:1,:]) * z[:,0,max_nl_shift:z.shape[2]-max_nl_shift] + nd.sin(self.gamma.data() * power[:,0:1,:]) * z[:,1,max_nl_shift:z.shape[2]-max_nl_shift]
        u = nd.concat(u, nd.cos(self.gamma.data() * power[:,0:1,:]) * z[:,1,max_nl_shift:z.shape[2]-max_nl_shift] - nd.sin(self.gamma.data() * power[:,0:1,:]) * z[:,0,max_nl_shift:z.shape[2]-max_nl_shift], dim=1)
        for i in range(1, int(self.chan * 2)):
            ind = int(i / 2)
            u = nd.concat(u, nd.cos(self.gamma.data() * power[:,ind:ind+1,:]) * z[:,2*i,max_nl_shift:z.shape[2]-max_nl_shift] + nd.sin(self.gamma.data() * power[:,ind:ind+1,:]) * z[:,2*i+1,max_nl_shift:z.shape[2]-max_nl_shift], dim=1)
            u = nd.concat(u, nd.cos(self.gamma.data() * power[:,ind:ind+1,:]) * z[:,2*i+1,max_nl_shift:z.shape[2]-max_nl_shift] - nd.sin(self.gamma.data() * power[:,ind:ind+1,:]) * z[:,2*i,max_nl_shift:z.shape[2]-max_nl_shift], dim=1)

        return u
    
    
class KerrActivationEnhanced_v2_4ch(gluon.Block): # Добавить сдвиги для случая разных коэффициентов для поляризаций и разная ширина на разных каналов
    def __init__(self, dimension, tensor_intra, tensor_inter1, tensor_inter2, tensor_inter3, nl_memory_size, nl_shifts, max_nl_shift, nl_coef=0.1, pol=2, strides=1, **kwargs):
        super(KerrActivationEnhanced_v2_4ch, self).__init__()
        chan = int(dimension/4)
        self.chan = chan
        self.nl_shifts = nl_shifts
        self.nl_memory_size = nl_memory_size
        self.max_nl_shift = max_nl_shift
        with self.name_scope():
            self.conv_intra = self.params.get('conv_intra', allow_deferred_init=True, shape=(chan,1,nl_memory_size[0]), init=CustomInit(tensor_intra))
            self.conv_inter1 = gluon.nn.Conv1D(channels=6, in_channels=6, groups=6, kernel_size=nl_memory_size[1], strides=strides, activation=None, use_bias=False, weight_initializer=CustomInit(tensor_inter1))
            self.conv_inter2 = gluon.nn.Conv1D(channels=4, in_channels=4, groups=4, kernel_size=nl_memory_size[2], strides=strides, activation=None, use_bias=False, weight_initializer=CustomInit(tensor_inter2))
            self.conv_inter3 = gluon.nn.Conv1D(channels=2, in_channels=2, groups=2, kernel_size=nl_memory_size[3], strides=strides, activation=None, use_bias=False, weight_initializer=CustomInit(tensor_inter3))
            self.gamma = self.params.get('gamma', shape=(1,), allow_deferred_init=True, init=mx.init.Constant(nl_coef))

    def forward(self, z):
        power = nd.square(z[:,0:1,:]) +  nd.square(z[:,1:2,:]) + nd.square(z[:,2:3,:]) +  nd.square(z[:,3:4,:])
        power_intra = power[:,:,self.nl_shifts[0,0,0]:power.shape[2]-self.nl_shifts[0,0,1]]
        power_inter1 = power[:,:,self.nl_shifts[0,1,0]:power.shape[2]-self.nl_shifts[0,1,1]]
        power_inter2 = power[:,:,self.nl_shifts[0,2,0]:power.shape[2]-self.nl_shifts[0,2,1]]
        power_inter3 = power[:,:,self.nl_shifts[0,3,0]:power.shape[2]-self.nl_shifts[0,3,1]]
                
        for i in range(1, self.chan):
            power = nd.square(z[:,4*i:4*i+1,:]) +  nd.square(z[:,4*i+1:4*i+2,:]) + nd.square(z[:,4*i+2:4*i+3,:]) +  nd.square(z[:,4*i+3:4*i+4,:])
            for j in range(self.chan):
                if i == j:
                    power_intra = nd.concat(power_intra, power[:,:,self.nl_shifts[i,i,0]:power.shape[2]-self.nl_shifts[i,i,1]], dim=1)
                elif abs(i-j) == 1:
                    power_inter1 = nd.concat(power_inter1, power[:,:,self.nl_shifts[i,j,0]:power.shape[2]-self.nl_shifts[i,j,1]], dim=1)
                elif abs(i-j) == 2:
                    power_inter2 = nd.concat(power_inter2, power[:,:,self.nl_shifts[i,j,0]:power.shape[2]-self.nl_shifts[i,j,1]], dim=1)
                else:
                    power_inter3 = nd.concat(power_inter3, power[:,:,self.nl_shifts[i,j,0]:power.shape[2]-self.nl_shifts[i,j,1]], dim=1)

        if sym_nonlin:
            tens_intra = nd.concat(self.conv_intra.data(), nd.flip(self.conv_intra.data(), axis=2)[:,:,1:], dim=2)
            power_intra = nd.Convolution(data = power_intra, weight = tens_intra, no_bias=True, num_filter=self.chan, kernel=2*self.nl_memory_size[0]-1, num_group=self.chan)
        else:
            power_intra = nd.Convolution(data = power_intra, weight = self.conv_intra.data(), no_bias=True, num_filter=self.chan, kernel=self.nl_memory_size[0], num_group=self.chan)
        power_inter1 = self.conv_inter1(power_inter1)
        power_inter2 = self.conv_inter2(power_inter2)
        power_inter3 = self.conv_inter3(power_inter3)
        power_inter = [power_inter1, power_inter2, power_inter3]
        
        power = power_intra
        power = power + nd.concat(power_inter1[:,1:2,:], power_inter1[:,0:1,:], power_inter2[:,0:1,:], power_inter3[:,0:1,:])
        power = power + nd.concat(power_inter2[:,2:3,:], power_inter1[:,3:4,:], power_inter1[:,2:3,:], power_inter2[:,1:2,:])
        power = power + nd.concat(power_inter3[:,1:2,:], power_inter2[:,3:4,:], power_inter1[:,5:6,:], power_inter1[:,4:5,:])
                       
        u = nd.cos(self.gamma.data() * power[:,0:1,:]) * z[:,0,self.max_nl_shift:z.shape[2]-self.max_nl_shift] + nd.sin(self.gamma.data() * power[:,0:1,:]) * z[:,1,self.max_nl_shift:z.shape[2]-self.max_nl_shift]
        u = nd.concat(u, nd.cos(self.gamma.data() * power[:,0:1,:]) * z[:,1,self.max_nl_shift:z.shape[2]-self.max_nl_shift] - nd.sin(self.gamma.data() * power[:,0:1,:]) * z[:,0,self.max_nl_shift:z.shape[2]-self.max_nl_shift], dim=1)
        for i in range(1, int(self.chan * 2)):
            ind = int(i / 2)
            u = nd.concat(u, nd.cos(self.gamma.data() * power[:,ind:ind+1,:]) * z[:,2*i,self.max_nl_shift:z.shape[2]-self.max_nl_shift] + nd.sin(self.gamma.data() * power[:,ind:ind+1,:]) * z[:,2*i+1,self.max_nl_shift:z.shape[2]-self.max_nl_shift], dim=1)
            u = nd.concat(u, nd.cos(self.gamma.data() * power[:,ind:ind+1,:]) * z[:,2*i+1,self.max_nl_shift:z.shape[2]-self.max_nl_shift] - nd.sin(self.gamma.data() * power[:,ind:ind+1,:]) * z[:,2*i,self.max_nl_shift:z.shape[2]-self.max_nl_shift], dim=1)

        return u
    
class ComplexConvSeq(nn.Sequential):
    def __init__(self, chan, filt_CDC_delay, filt_CDC_last_delay, filt_FD_delay, step_output_shifts, last_output_shifts, last_step_length, FD_step_length, **kwargs):
        super(ComplexConvSeq, self).__init__(**kwargs)
        self.chan = chan
        self.step_shifts = step_output_shifts
        self.last_shifts = last_output_shifts
        self.filt_CDC_delay = filt_CDC_delay
        self.filt_CDC_last_delay = filt_CDC_last_delay
        self.filt_FD_delay = filt_FD_delay
        self.last_step_length = last_step_length
        self.FD_step_length = last_step_length
    def forward(self, z):
        v = self._children['0'](z)
        u = v
        for i in range(1, num_step):
            u = self._children[str(i)](u)
            if manual_shift:
                v = nd.concat(v[:,:,self.step_shifts[0]+self.filt_CDC_delay:v.shape[2]-self.filt_CDC_delay-self.step_shifts[1]], u, dim=0)
            else:
                v = nd.concat(v[:,:,self.filt_CDC_delay:v.shape[2]-self.filt_CDC_delay], u, dim=0)
        if self.last_step_length > 0:
            u = self._children[str(num_step)](u)
            if manual_shift:
                v = nd.concat(v[:,:,self.last_shifts[0]+self.filt_CDC_last_delay:v.shape[2]-self.filt_CDC_last_delay-self.last_shifts[1]], u, dim=0)
            else:
                v = nd.concat(v[:,:,self.filt_CDC_last_delay:v.shape[2]-self.filt_CDC_last_delay], u, dim=0)
        if self.FD_step_length > 0:
            u = self._children[str(len(self._children.items())-1)](u)
            v = nd.concat(v[:,:,self.filt_FD_delay:v.shape[2]-self.filt_FD_delay], u, dim=0)
        return v

# 5 CDC and FD FIR filters

In [ ]:
# Функция для вычисления комплексного фильтра для компенсации накопленной дисперсии на одном шаге
def get_CDCfilt_step(data_real, width, length, shift, output_shift, dir_path, train_print=True):
    if sym_filt:
        delay = width - 1
    else:
        delay = int((width-1)/2)
    data = data_real[0,:] + 1j * data_real[1,:]
    size = len(data)
    filt_Re = nd.empty((channels, 1, width), dtype=real_t)
    filt_Im = nd.empty((channels, 1, width), dtype=real_t)
    
    f = open(dir_path + 'filter_train.dat', 'a+')
    
    for i in range(channels):
        data_CDC = cd_operator(data, size, length, Fn[channel_numbers[i]], 'b')
        if manual_shift:
            data_CDC = data_CDC[delay+output_shift[0]:size-delay-output_shift[1]]
        else:
            data_CDC = data_CDC[delay:size-delay]
            
        net = gluon.nn.Sequential()
        with net.name_scope():
            net.add(ComplexConvRandom(kernel_size=width, dimension=2, shift=shift[i:i+1,:], pol=1, enable_shift=manual_shift, sym=sym_filt))

        net.initialize(mx.init.Normal(sigma=0.05), ctx=ctx)
        trainer = gluon.Trainer(net.collect_params(), 'Adam', {'learning_rate': lrate_CDC})
        mse = gluon.loss.L2Loss()

        features = nd.empty((1, 2, size))
        features[:,0,:] = nd.array(np.real(data))
        features[:,1,:] = nd.array(np.imag(data))
        features = features.as_in_context(ctx)
        if manual_shift:
            labels = nd.empty((1,2,size-2*delay-abs(np.sum(output_shift))))
        else:
            labels = nd.empty((1,2,size-2*delay))
        labels[:,0,:] = nd.array(np.real(data_CDC))
        labels[:,1,:] = nd.array(np.imag(data_CDC))
        labels = labels.as_in_context(ctx)
        
        lossCond = True
        epoch = 0
        prev_loss = 100
        while lossCond and epoch < 10000 and prev_loss > 1e-4:
            epoch += 1
            tic = time.time()
            train_loss = nd.zeros(1, ctx=ctx)
            with autograd.record():
                output = net(features)
                loss = mse(output, labels)
            loss.backward()
            trainer.step(size)
            train_loss += loss.mean().asscalar()
            loss.wait_to_read()
            if epoch % 100 == 0:
                if train_print:
                    print('CDC filter step -- channel', channel_numbers[i], 'epoch', epoch, '-- loss', train_loss.asscalar(), '-- time', time.time()-tic)
                f.write('CDC filter step -- channel {} epoch {} -- loss {} -- time {}\n'.format(channel_numbers[i], epoch, train_loss.asscalar(), time.time()-tic))
                lossCond = abs(train_loss.asscalar() - prev_loss) / prev_loss > 1e-6
                prev_loss = train_loss.asscalar()

        params = net.collect_params()
        filt_Re[i,:,:] = params[list(params)[0]].data().reshape(width,).asnumpy()
        filt_Im[i,:,:] = params[list(params)[1]].data().reshape(width,).asnumpy()
        
        f.write('\n')
        
    f.write('\n\n')
    f.close()
    return [filt_Re, filt_Im]


def get_CDCfilt_step_1ch(data_real, width, length, shift, output_shift, dir_path, train_print=True):
    if sym_filt:
        delay = width - 1
    else:
        delay = int((width-1)/2)
    data = data_real[0,:] + 1j * data_real[1,:]
    size = len(data)
    filt_Re = nd.empty((channels, 1, width), dtype=real_t)
    filt_Im = nd.empty((channels, 1, width), dtype=real_t)
    
    f = open(dir_path + 'filter_train.dat', 'a+')
    
    for i in range(channels):
        data_CDC = cd_operator(data, length, 1, Fn[channel_numbers[i]], 'b')
        if manual_shift:
            data_CDC = data_CDC[delay+output_shift[0]:size-delay-output_shift[1]]
        else:
            data_CDC = data_CDC[delay:size-delay]
            
        net = gluon.nn.Sequential()
        with net.name_scope():
            net.add(ComplexConvRandom(kernel_size=width, dimension=2, shift=shift[i:i+1,:], pol=1, enable_shift=manual_shift, sym=sym_filt))

        net.initialize(mx.init.Normal(sigma=0.05), ctx=ctx)
        trainer = gluon.Trainer(net.collect_params(), 'Adam', {'learning_rate': lrate_CDC})
        mse = gluon.loss.L2Loss()

        features = nd.empty((1, 2, size))
        features[:,0,:] = nd.array(np.real(data))
        features[:,1,:] = nd.array(np.imag(data))
        features = features.as_in_context(ctx)
        if manual_shift:
            labels = nd.empty((1,2,size-2*delay-abs(np.sum(output_shift))))
        else:
            labels = nd.empty((1,2,size-2*delay))
        labels[:,0,:] = nd.array(np.real(data_CDC))
        labels[:,1,:] = nd.array(np.imag(data_CDC))
        labels = labels.as_in_context(ctx)
        
        lossCond = True
        epoch = 0
        prev_loss = 100
        while lossCond and epoch < 10000 and prev_loss > 1e-4:
            epoch += 1
            tic = time.time()
            train_loss = nd.zeros(1, ctx=ctx)
            with autograd.record():
                output = net(features)
                loss = mse(output, labels)
            loss.backward()
            trainer.step(size)
            train_loss += loss.mean().asscalar()
            loss.wait_to_read()
            if epoch % 100 == 0:
                if train_print:
                    print('CDC filter step -- channel', channel_numbers[i], 'epoch', epoch, '-- loss', train_loss.asscalar(), '-- time', time.time()-tic)
                f.write('CDC filter step -- channel {} epoch {} -- loss {} -- time {}\n'.format(channel_numbers[i], epoch, train_loss.asscalar(), time.time()-tic))
                lossCond = abs(train_loss.asscalar() - prev_loss) / prev_loss > 1e-6
                prev_loss = train_loss.asscalar()

        params = net.collect_params()
        filt_Re[i,:,:] = params[list(params)[0]].data().reshape(width,).asnumpy()
        filt_Im[i,:,:] = params[list(params)[1]].data().reshape(width,).asnumpy()
        
        f.write('\n')
        
    f.write('\n\n')
    f.close()
    return [filt_Re, filt_Im]


# Функция для вычисления комлексного фильтра с дробным сдвигом
def get_FDfilt(data_real, width, length, dir_path, train_print=True):
    delay = int((width-1)/2)
    data = data_real[0,:] + 1j * data_real[1,:]
    size = len(data)
    filtFD_Re = nd.empty((channels, 1, width), dtype=real_t)
    filtFD_Im = nd.empty((channels, 1, width), dtype=real_t)
    
    f = open(dir_path + 'filter_train.dat', 'a+')
    
    for i in range(channels):
        data_CDC = cd_operator(data, size, length, Fn[channel_numbers[i]], 'b')
        data_CDC = data_CDC[delay:size-delay]
    
        net = gluon.nn.Sequential()
        with net.name_scope():
            net.add(ComplexConvRandom(kernel_size=width, dimension=2, pol=1, enable_shift=False, sym=False))
    
        net.initialize(mx.init.Normal(sigma=0.05), ctx=ctx)
        trainer = gluon.Trainer(net.collect_params(), 'Adam', {'learning_rate': lrate_CDC})
        mse = gluon.loss.L2Loss()
    
        features = nd.empty((1, 2, size))
        features[:,0,:] = nd.array(np.real(data))
        features[:,1,:] = nd.array(np.imag(data))
        features = features.as_in_context(ctx)
        labels = nd.empty((1,2,size-2*delay))
        labels[:,0,:] = nd.array(np.real(data_CDC))
        labels[:,1,:] = nd.array(np.imag(data_CDC))
        labels = labels.as_in_context(ctx)

        lossCond = True
        epoch = 0
        prev_loss = 100
        while lossCond and epoch < 10000 and prev_loss > 1e-5:
            epoch += 1
            tic = time.time()
            train_loss = nd.zeros(1, ctx=ctx)
            with autograd.record():
                output = net(features)
                loss = mse(output, labels)
            loss.backward()
            trainer.step(size)
            train_loss += loss.mean().asscalar()
            loss.wait_to_read()
            if epoch % 100 == 0:
                if train_print:
                    print('FD filter -- channel', channel_numbers[i], 'epoch', epoch, '-- loss', train_loss.asscalar(), '-- time', time.time()-tic)
                f.write('FD filter step -- channel {} epoch {} -- loss {} -- time {}\n'.format(channel_numbers[i], epoch, train_loss.asscalar(), time.time()-tic))
                lossCond = abs(train_loss.asscalar() - prev_loss) / prev_loss > 1e-6
                prev_loss = train_loss.asscalar()

        params = net.collect_params()
        filtFD_Re[i,:,:] = params[list(params)[0]].data().reshape(width,).asnumpy()
        filtFD_Im[i,:,:] = params[list(params)[1]].data().reshape(width,).asnumpy()
        
        f.write('\n')
        
    f.write('\n\n')
    f.close()
    return [filtFD_Re, filtFD_Im]

def get_CDCfilt_JO(data_real, filt, filt_CDC_delay, step_length, last_step_length, filt_CDC_last_delay, FD_step_length, filt_FD_delay, step_shifts, last_shifts, step_output_shifts, last_output_shifts,
                   full_output_shifts, max_full_shift, dir_path, train_print=True):

    data = data_real[0,:] + 1j * data_real[1,:]
    size = len(data)
    
    delay = num_step * filt_CDC_delay
    if last_step_length > 0:
        delay += filt_CDC_last_delay
    if FD_step_length > 0:
        delay += filt_FD_delay
    
    filt_CDC_width = filt[0].shape[2]
    if last_step_length > 0:
        filt_CDC_last_width = filt[-3].shape[2]
    if FD_step_length > 0:
        filt_FD_width = filt[-1].shape[2]
    
    filtCDC_Re = nd.empty((channels, num_step, filt_CDC_width), dtype=real_t)
    filtCDC_Im = nd.empty((channels, num_step, filt_CDC_width), dtype=real_t)
    if last_step_length > 0:
        filtCDC_last_Re = nd.empty((channels, 1, filt_CDC_last_width), dtype=real_t)
        filtCDC_last_Im = nd.empty((channels, 1, filt_CDC_last_width), dtype=real_t)
    if FD_step_length > 0:
        filtFD_Re = nd.empty((channels, 1, filt_FD_width), dtype=real_t)
        filtFD_Im = nd.empty((channels, 1, filt_FD_width), dtype=real_t)
    
    f = open(dir_path + 'filter_train.dat', 'a+')
    
    for i in range(channels):
        
        netJO = ComplexConvSeq(i, filt_CDC_delay, filt_CDC_last_delay, filt_FD_delay, step_output_shifts, last_output_shifts, last_step_length, FD_step_length)
        for j in range(num_step):
            netJO.add(ComplexConvPredefined(weightsRe=filt[0][i,:,:], weightsIm=filt[1][i,:,:], kernel_size=filt_CDC_width, shift=step_shifts[i:i+1,:], dimension=2, pol=1, enable_shift=manual_shift, sym=sym_filt))
        if last_step_length > 0:
            netJO.add(ComplexConvPredefined(weightsRe=filt[2][i,:,:], weightsIm=filt[3][i,:,:], kernel_size=filt_CDC_last_width, shift=last_shifts[i:i+1,:], dimension=2, pol=1, enable_shift=manual_shift, sym=sym_filt))
        if FD_step_length > 0:
            netJO.add(ComplexConvPredefined(weightsRe=filt[len(filt)-2][i,:,:], weightsIm=filt[len(filt)-1][i,:,:], kernel_size=filt_FD_width, dimension=2, pol=1, enable_shift=False, sym=False))
        
        netJO.initialize(mx.init.Normal(sigma=0.05), ctx=ctx)
        trainer = gluon.Trainer(netJO.collect_params(), 'Adam', {'learning_rate': lrate_CDC})
        mse = gluon.loss.L2Loss()

        features = nd.empty((1, 2, size))
        features[:,0,:] = nd.array(np.real(data))
        features[:,1,:] = nd.array(np.imag(data))
        features = features.as_in_context(ctx)
        
        label_dimension = num_step
        if last_step_length > 0:
            label_dimension += 1
        if FD_step_length > 0:
            label_dimension += 1
        if manual_shift:
            labels = nd.empty((label_dimension,2,size-2*delay-max_full_shift))
        else:
            labels = nd.empty((label_dimension,2,size-2*delay))
            
        for j in range(num_step):
            data_CDC = cd_operator(data, size, (j + 1) * step_length, Fn[channel_numbers[i]], 'b')
            if manual_shift:
                data_CDC = data_CDC[delay+full_output_shifts[0]:size-(delay+full_output_shifts[1])]
            else:
                data_CDC = data_CDC[delay:size-delay]
            labels[j,0,:] = nd.array(np.real(data_CDC))
            labels[j,1,:] = nd.array(np.imag(data_CDC))
        if last_step_length > 0:
            data_CDC = cd_operator(data, size, (j + 1) * step_length + last_step_length, Fn[channel_numbers[i]], 'b')
            if manual_shift:
                data_CDC = data_CDC[delay+full_output_shifts[0]:size-(delay+full_output_shifts[1])]
            else:
                data_CDC = data_CDC[delay:size-delay]
            labels[num_step,0,:] = nd.array(np.real(data_CDC))
            labels[num_step,1,:] = nd.array(np.imag(data_CDC))
        if FD_step_length > 0:
            data_CDC = cd_operator(data, size, fiberL, Fn[channel_numbers[i]], 'b')
            if manual_shift:
                data_CDC = data_CDC[delay+full_output_shifts[0]:size-(delay+full_output_shifts[1])]
            else:
                data_CDC = data_CDC[delay:size-delay]
            labels[label_dimension-1,0,:] = nd.array(np.real(data_CDC))
            labels[label_dimension-1,1,:] = nd.array(np.imag(data_CDC))
        labels = labels.as_in_context(ctx)
        
        lossCond = True
        epoch = 0
        prev_loss = 10
        while lossCond and epoch < 2000:
            epoch += 1
            tic = time.time()
            train_loss = nd.zeros(1, ctx=ctx)
            with autograd.record():
                output = netJO(features)
                loss = mse(output, labels)
            loss.backward()
            trainer.step(size)
            train_loss += loss.mean().asscalar()
            loss.wait_to_read()
            if epoch % 100 == 0:
                if train_print:
                    print('CDC filter sequence -- channel', channel_numbers[i], 'epoch', epoch, '-- loss', train_loss.asscalar(), '-- time', time.time()-tic)
                f.write('CDC filter sequence -- channel {} epoch {} -- loss {} -- time {}\n'.format(channel_numbers[i], epoch, train_loss.asscalar(), time.time()-tic))
                lossCond = train_loss.asscalar() > 1e-4
                if prev_loss < train_loss.asscalar():
                    break;
                else:
                    prev_loss = train_loss.asscalar()
                    
        params = netJO.collect_params()
        for j in range(num_step):
            filtCDC_Re[i,j,:] = params[list(params)[2*j]].data().reshape(filt_CDC_width,).asnumpy()
            filtCDC_Im[i,j,:] = params[list(params)[2*j+1]].data().reshape(filt_CDC_width,).asnumpy()
        if last_step_length > 0:
            filtCDC_last_Re[i,:,:] = params[list(params)[2*num_step]].data().reshape(filt_CDC_last_width,).asnumpy()
            filtCDC_last_Im[i,:,:] = params[list(params)[2*num_step+1]].data().reshape(filt_CDC_last_width,).asnumpy()
        if FD_step_length > 0:
            filtFD_Re[i,:,:] = params[list(params)[len(list(params))-1]].data().reshape(filt_FD_width,).asnumpy()
            filtFD_Im[i,:,:] = params[list(params)[len(list(params))-2]].data().reshape(filt_FD_width,).asnumpy()
         
        f.write('\n')
    
    f.write('\n\n')
    f.close()
    filters = [filtCDC_Re, filtCDC_Im]
    if last_step_length > 0:
            filters = filters + [filtCDC_last_Re, filtCDC_last_Im]
    if FD_step_length > 0:
            filters = filters + [filtFD_Re, filtFD_Im]
    return filters


def get_CDCfilt_JO_1ch(data_real, filt, filt_CDC_delay, step_length, last_step_length, filt_CDC_last_delay, FD_step_length, filt_FD_delay, step_shifts, last_shifts, step_output_shifts, last_output_shifts,
                   full_output_shifts, max_full_shift, dir_path, train_print=True):

    data = data_real[0,:] + 1j * data_real[1,:]
    size = len(data)
    
    delay = num_step * filt_CDC_delay
    if last_step_length > 0:
        delay += filt_CDC_last_delay
    if FD_step_length > 0:
        delay += filt_FD_delay
    
    filt_CDC_width = filt[0].shape[2]
    if last_step_length > 0:
        filt_CDC_last_width = filt[-3].shape[2]
    if FD_step_length > 0:
        filt_FD_width = filt[-1].shape[2]
    
    filtCDC_Re = nd.empty((channels, num_step, filt_CDC_width), dtype=real_t)
    filtCDC_Im = nd.empty((channels, num_step, filt_CDC_width), dtype=real_t)
    if last_step_length > 0:
        filtCDC_last_Re = nd.empty((channels, 1, filt_CDC_last_width), dtype=real_t)
        filtCDC_last_Im = nd.empty((channels, 1, filt_CDC_last_width), dtype=real_t)
    if FD_step_length > 0:
        filtFD_Re = nd.empty((channels, 1, filt_FD_width), dtype=real_t)
        filtFD_Im = nd.empty((channels, 1, filt_FD_width), dtype=real_t)
    
    f = open(dir_path + 'filter_train.dat', 'a+')
    
    for i in range(channels):
        
        netJO = ComplexConvSeq(i, filt_CDC_delay, filt_CDC_last_delay, filt_FD_delay, step_output_shifts, last_output_shifts, last_step_length, FD_step_length)
        for j in range(num_step):
            netJO.add(ComplexConvPredefined(weightsRe=filt[0][i,:,:], weightsIm=filt[1][i,:,:], kernel_size=filt_CDC_width, shift=step_shifts[i:i+1,:], dimension=2, pol=1, enable_shift=manual_shift, sym=sym_filt))
        if last_step_length > 0:
            netJO.add(ComplexConvPredefined(weightsRe=filt[2][i,:,:], weightsIm=filt[3][i,:,:], kernel_size=filt_CDC_last_width, shift=last_shifts[i:i+1,:], dimension=2, pol=1, enable_shift=manual_shift, sym=sym_filt))
        if FD_step_length > 0:
            netJO.add(ComplexConvPredefined(weightsRe=filt[len(filt)-2][i,:,:], weightsIm=filt[len(filt)-1][i,:,:], kernel_size=filt_FD_width, dimension=2, pol=1, enable_shift=False, sym=False))
        
        netJO.initialize(mx.init.Normal(sigma=0.05), ctx=ctx)
        trainer = gluon.Trainer(netJO.collect_params(), 'Adam', {'learning_rate': lrate_CDC})
        mse = gluon.loss.L2Loss()

        features = nd.empty((1, 2, size))
        features[:,0,:] = nd.array(np.real(data))
        features[:,1,:] = nd.array(np.imag(data))
        features = features.as_in_context(ctx)
        
        label_dimension = num_step
        if last_step_length > 0:
            label_dimension += 1
        if FD_step_length > 0:
            label_dimension += 1
        if manual_shift:
            labels = nd.empty((label_dimension,2,size-2*delay-max_full_shift))
        else:
            labels = nd.empty((label_dimension,2,size-2*delay))
            
        for j in range(num_step):
            data_CDC = cd_operator(data, (j + 1) * step_length, 1, Fn[channel_numbers[i]], 'b')
            if manual_shift:
                data_CDC = data_CDC[delay+full_output_shifts[0]:size-(delay+full_output_shifts[1])]
            else:
                data_CDC = data_CDC[delay:size-delay]
            labels[j,0,:] = nd.array(np.real(data_CDC))
            labels[j,1,:] = nd.array(np.imag(data_CDC))
        if last_step_length > 0:
            data_CDC = cd_operator(data, (j + 1) * step_length + last_step_length, 1, Fn[channel_numbers[i]], 'b')
            if manual_shift:
                data_CDC = data_CDC[delay+full_output_shifts[0]:size-(delay+full_output_shifts[1])]
            else:
                data_CDC = data_CDC[delay:size-delay]
            labels[num_step,0,:] = nd.array(np.real(data_CDC))
            labels[num_step,1,:] = nd.array(np.imag(data_CDC))
        if FD_step_length > 0:
            data_CDC = cd_operator(data, fiberL, 1, Fn[channel_numbers[i]], 'b')
            if manual_shift:
                data_CDC = data_CDC[delay+full_output_shifts[0]:size-(delay+full_output_shifts[1])]
            else:
                data_CDC = data_CDC[delay:size-delay]
            labels[label_dimension-1,0,:] = nd.array(np.real(data_CDC))
            labels[label_dimension-1,1,:] = nd.array(np.imag(data_CDC))
        labels = labels.as_in_context(ctx)
        
        lossCond = True
        epoch = 0
        prev_loss = 10
        while lossCond and epoch < 2000:
            epoch += 1
            tic = time.time()
            train_loss = nd.zeros(1, ctx=ctx)
            with autograd.record():
                output = netJO(features)
                loss = mse(output, labels)
            loss.backward()
            trainer.step(size)
            train_loss += loss.mean().asscalar()
            loss.wait_to_read()
            if epoch % 100 == 0:
                if train_print:
                    print('CDC filter sequence -- channel', channel_numbers[i], 'epoch', epoch, '-- loss', train_loss.asscalar(), '-- time', time.time()-tic)
                f.write('CDC filter sequence -- channel {} epoch {} -- loss {} -- time {}\n'.format(channel_numbers[i], epoch, train_loss.asscalar(), time.time()-tic))
                lossCond = train_loss.asscalar() > 1e-4
                if prev_loss < train_loss.asscalar():
                    break;
                else:
                    prev_loss = train_loss.asscalar()
                    
        params = netJO.collect_params()
        for j in range(num_step):
            filtCDC_Re[i,j,:] = params[list(params)[2*j]].data().reshape(filt_CDC_width,).asnumpy()
            filtCDC_Im[i,j,:] = params[list(params)[2*j+1]].data().reshape(filt_CDC_width,).asnumpy()
        if last_step_length > 0:
            filtCDC_last_Re[i,:,:] = params[list(params)[2*num_step]].data().reshape(filt_CDC_last_width,).asnumpy()
            filtCDC_last_Im[i,:,:] = params[list(params)[2*num_step+1]].data().reshape(filt_CDC_last_width,).asnumpy()
        if FD_step_length > 0:
            filtFD_Re[i,:,:] = params[list(params)[len(list(params))-1]].data().reshape(filt_FD_width,).asnumpy()
            filtFD_Im[i,:,:] = params[list(params)[len(list(params))-2]].data().reshape(filt_FD_width,).asnumpy()
         
        f.write('\n')
    
    f.write('\n\n')
    f.close()
    filters = [filtCDC_Re, filtCDC_Im]
    if last_step_length > 0:
            filters = filters + [filtCDC_last_Re, filtCDC_last_Im]
    if FD_step_length > 0:
            filters = filters + [filtFD_Re, filtFD_Im]
    return filters